In [ ]:
# Post-integration analysis of an intestine reference dataset (signature gene lists).
import scanpy as sc

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import anndata as ad

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)

In [ ]:
# Load data
adata = sc.read('/home/yanjun3/cell2sentence/intestine/annotated_healthy_disease.h5ad')

In [ ]:
adata.obs_names_make_unique()

In [ ]:
import seaborn as sns
p1 = sns.displot(adata.obs["total_counts"], bins=100, kde=False)
p2 = sc.pl.violin(adata, "pct_counts_mt")
p3 = sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt")

In [ ]:
# Neighbours, UMAP and clustering
sc.pp.neighbors(adata, use_rep="X_scVI", key_added="neighbors_scvi")
sc.tl.umap(adata, neighbors_key="neighbors_scvi", key_added="X_umap_scvi")
sc.pl.embedding(
    adata,
    ncols = 1,
    color = ['sample', 'age', 'position', 'Race', 'sex'], size = 100,
    basis = 'X_umap_scvi')

In [ ]:
sc.pp.neighbors(adata, use_rep="X_scANVI", key_added="neighbors_scanvi")
sc.tl.umap(adata, neighbors_key="neighbors_scanvi", key_added="X_umap_scanvi")
sc.pl.embedding(
    adata,
    ncols = 1,
    color = ['sample', 'age', 'position', 'Race', 'sex'], size = 100,
    basis = 'X_umap_scanvi')

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score
scores = {
    "scVI": {
        "Batch (sample)": silhouette_score(adata.obsm["X_scVI"], adata.obs["sample"].astype(str)),
        "Cell Type": silhouette_score(adata.obsm["X_scVI"], adata.obs["celltypist_labels"].astype(str))
    },
    "scANVI": {
        "Batch (sample)": silhouette_score(adata.obsm["X_scANVI"], adata.obs["sample"].astype(str)),
        "Cell Type": silhouette_score(adata.obsm["X_scANVI"], adata.obs["celltypist_labels"].astype(str))
    }
}

# Create DataFrame
df_scores = pd.DataFrame(scores).T

# Plot results
df_scores.plot(kind="bar", figsize=(8, 5), rot=0)
plt.title("Silhouette Scores: scVI vs scANVI")
plt.ylabel("Silhouette Score")
plt.axhline(0, linestyle="--", color="gray")
plt.tight_layout()
plt.show()


In [ ]:
import scib
adata_scvi = adata.copy()
adata_scanvi = adata.copy()

In [ ]:
sc.pp.neighbors(adata, use_rep="X_scVI")
metrics_scvi = scib.metrics.metrics(
    adata_scvi, adata,
    batch_key="sample",
    label_key="celltypist_labels",
    embed="X_scVI",
    silhouette_=True,
    hvg_score_=False,
    graph_conn_=True,
    pcr_=True,
    isolated_labels_asw_=True
)


In [ ]:
sc.pp.neighbors(adata, use_rep="X_scANVI")
metrics_scanvi = scib.metrics.metrics(
    adata_scanvi, adata,
    batch_key="sample",
    label_key="celltypist_labels",
    embed="X_scANVI",
    silhouette_=True,
    hvg_score_=False,
    graph_conn_=True,
    pcr_=True,
    isolated_labels_asw_=True
)


In [ ]:
# Concatenate metrics results
metrics = pd.concat(
    [metrics_scvi, metrics_scanvi],
    axis="columns",
)
# Set methods as column names
metrics = metrics.set_axis(
    ["scVI", "scANVI"], axis="columns"
)
# Select only the fast metrics
metrics = metrics.loc[
    [
        "ASW_label",
        "ASW_label/batch",
        "PCR_batch",
        "isolated_label_silhouette",
        "graph_conn",
    ],
    :,
]
# Transpose so that metrics are columns and methods are rows
metrics = metrics.T
metrics


In [ ]:
sc.pp.neighbors(adata, use_rep="X_scVI")

In [ ]:
sc.tl.leiden(adata, resolution=0.6,key_added='scvi_06')
sc.pl.embedding(
    adata,
    ncols = 1,
    color = ['scvi_06', 'celltypist_labels'], size = 100,
    basis = 'X_umap_scvi')

In [ ]:
p = sc.pl.violin(
    adata,
    keys=["total_counts", "n_genes_by_counts", "pct_counts_mt"],
    groupby="scvi_06",
    jitter=0.4,
    multi_panel=True
)


In [ ]:
marker_dict = {
    "Classical Monocytes": ['S100A8', 'S100A9', 'VCAN', 'CD14', 'LYZ'],
    "Non-Classical Monocytes": ['FCGR3A', 'ITGAX'],
    "Macrophages": ['C1QA', 'C1QB', 'C1QC', 'APOE',  'STAB1', 'LYVE1',  'CD68'],
    "Inflammatory Macrophages": ['HLA-DRA', 'HLA-A'
                                 ],
    "LYVE1+ TRM": ['LYVE1', 'SELENOP', 'F13A1', 'MRC1', 'TIMD4', 'FOLR2'],
    "SPP1+ Macrophages": ['SPP1', 'MMP9', 'FN1', 'CHI3L1', 'FABP5', 'TREM2'],
    "Cycling Myeloid": ['CD33', 'IL3RA', 'ITGAX', 'THBD', 'CD1C'],
    "pan DC" : ['CLEC10A', 'ITGAM'], # CD11C
    "Other Macrophages": ['FABP4', 'PPARG', 'PLIN2', 'APOC1', 'APOE']
}


sc.pl.dotplot(
    adata, 
    var_names=marker_dict,
    groupby='scvi_06',
    dot_max = 0.4,
    standard_scale='var', 
    color_map='RdBu_r' ,
    
    save = 'scvi_06.png'
)

In [ ]:

sc.pl.dotplot(
    adata, 
    var_names=marker_dict,
    groupby='annotation',
    dot_max = 0.4,
    standard_scale=False, 
    color_map='RdBu_r' ,
    save = 'scvi_06_noscale.png',
    vmin = 0,
    vmax = 1.5
)

In [ ]:
# Differential expression
sc.tl.rank_genes_groups(
    adata,
    groupby='scvi_06',
    method='wilcoxon')


In [ ]:
sc.pl.rank_genes_groups_matrixplot(
    adata,
    n_genes=30,
    groupby='scvi_06',               
    key='rank_genes_groups',                            
    cmap='RdBu_r',                      
    standard_scale='var',           
    figsize=(10, 80),
    swap_axes=True,
save  = 'scvi_06')

In [ ]:
cluster_annotation = {
    "0": "Transitional Classical Monocytes",
    "1": "SPP1⁺ MMP⁺ Macrophages",
    "2": "LYVE1⁺ Tissue-Resident Macrophages",
    "3": "APOE⁺ APOC1⁺ Macrophages",
    "4": "Transitional Classical Monocytes",
    "5": "Cycling Classical Monocytes",
    "6": "CLEC10A⁺ Dendritic Cells",
    "7": "Transitional Classical Monocytes",
    "8": "Macrophages",
    "9": "Macrophages",
    "10": "Cycling Classical Monocytes",
    "11": "Apoptotic Cells"
}

adata.obs["scvi_06_annotation"] = adata.obs["scvi_06"].map(cluster_annotation)

In [ ]:
sc.pl.embedding(
    adata,
    ncols = 1,
    color = ['scvi_06'], size = 100, legend_loc = 'on data',
    basis = 'X_umap_scvi', save = 'umap.png')

In [ ]:
sc.pl.embedding(
    adata,
    ncols = 1,
    color = ['annotation'], size = 100, 
    basis = 'X_umap_scvi', save = 'umap1.png')

In [ ]:
result = adata.uns['rank_genes_groups']
groups = result['names'].dtype.names
df = pd.DataFrame(
    {f"{group}_{key}": result[key][group]
     for group in groups for key in ['names', 'scores', 'pvals', 'pvals_adj', 'logfoldchanges']}
)
df.to_csv('/home/yc/Desktop/atlas/intestine/deg/scvi06.csv', index=False)

In [ ]:
group_col = "annotation"  
# Count and normalize within disease
df_disease = (
    adata.obs[[group_col, "disease"]]
    .groupby(["disease", group_col])
    .size()
    .unstack(fill_value=0)
)
percent_disease = df_disease.div(df_disease.sum(axis=1), axis=0) * 100
percent_disease = percent_disease.iloc[::-1]

# Plot
fig1 = plt.figure(figsize=(10, 3))
ax1 = percent_disease.plot(kind='barh', stacked=True, ax=fig1.gca(), colormap='tab20')

plt.title(f"Proportion of {group_col} across Disease")
plt.xlabel("Percentage (%)")
plt.ylabel("Disease")
plt.legend(title=group_col, bbox_to_anchor=(1.05, 1), loc='upper left', prop={'size': 10})  # 👈 Here!
plt.grid(False)
plt.tight_layout()
plt.savefig('propotion_disease.png')
plt.show


In [ ]:
df_loc = (
    adata.obs[[group_col, "position"]]
    .groupby(["position", group_col])
    .size()
    .unstack(fill_value=0)
)
percent_loc = df_loc.div(df_loc.sum(axis=1), axis=0) * 100
percent_loc = percent_loc.iloc[::-1]

# Plot
fig2 = plt.figure(figsize=(10, 3))
ax2 = percent_loc.plot(kind='barh', stacked=True, ax=fig2.gca(), colormap='tab20')

plt.title(f"Proportion of {group_col} across Location")
plt.xlabel("Percentage (%)")
plt.ylabel("position")
plt.legend(title=group_col, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(False)
plt.tight_layout()
plt.savefig('/home/yc/Desktop/atlas/intestine/figures/propotion_location.png')

plt.show()


In [ ]:
df_loc = (
    adata.obs[[group_col, "position"]]
    .groupby(["position", group_col])
    .size()
    .unstack(fill_value=0)
)
percent_loc = df_loc.div(df_loc.sum(axis=1), axis=0) * 100
percent_loc = percent_loc.iloc[::-1]

# Plot
fig2 = plt.figure(figsize=(12, 3))
ax2 = percent_loc.plot(kind='barh', stacked=True, ax=fig2.gca(), colormap='tab20')

plt.title(f"Proportion of {group_col} across Location")
plt.xlabel("Percentage (%)")
plt.ylabel("position")
plt.legend(title=group_col, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(False)
plt.tight_layout()
plt.savefig('/home/yc/Desktop/atlas/intestine/figures/propotion_location.png')
plt.show()

In [ ]:
adata

In [ ]:
# Save result
adata.write("/home/yc/Desktop/atlas/intestine/annotated_healthy_disease.h5ad")

In [ ]:
bulktop50 = pd.read_csv('/home/yanjun3/atlas/signatures/bulktop50.csv', header = None)
mac4top50 = pd.read_csv('/home/yanjun3/atlas/signatures/mac4top50.csv', header = None)
mac7top50 = pd.read_csv('/home/yanjun3/atlas/signatures/mac7top100.csv', header = None)
eraslanhlatop50 = pd.read_csv('/home/yanjun3/atlas/signatures/eralanhlatop50.csv', header = None)

In [ ]:
# Gene scoring
gene_list = bulktop50[0].tolist()
sc.tl.score_genes(adata, gene_list, score_name='bulktop50_score')

In [ ]:
sc.tl.score_genes(adata, mac4top50[0].tolist(), score_name='mac4top50_score')
sc.tl.score_genes(adata, mac7top50[1].iloc[0:50].tolist(), score_name='mac7top50_score')
sc.tl.score_genes(adata, eraslanhlatop50[0].tolist(), score_name='eraslanhlatop50_score')

In [ ]:
sc.pl.embedding(adata, color = ['bulktop50_score'], ncols = 1,
          vmin = 0, vmax = 0.4, basis = 'X_umap_scvi', save = 'bulkscore.png')

In [ ]:
sc.pl.embedding(adata, color = ['mac4top50_score' ,
                           'mac7top50_score'], ncols = 1,
          vmin = 0, vmax = 1, basis = 'X_umap_scvi', save = 'scscore.png')

In [ ]:
sc.pl.embedding(adata, color = [ 'eraslanhlatop50_score'], ncols = 1,
          vmin = 0, vmax = 0.2, basis = 'X_umap_scvi', save = 'erscore.png')

In [ ]:
adata1 = sc.read('/home/yanjun2/scanpy/8_figures/inte_palantir.h5ad')

In [ ]:
l1_genes = [
    "SELENOP", "SLC40A1", "C1QA", "STAB1", "MS4A7", "LGMN", "FGL2", "C1QC", "MT-ND1", "FOS", "CST3", "MT-ND2",
    "MAF", "DAB2", "RGS1", "MT-CO3", "ITM2B", "MS4A6A", "MPEG1", "FCGRT", "SLCO2B1", "A2M", "MT-ND4", "C1QB",
    "MT-ND3", "CD74", "JUND", "FOSB", "HERPUD1", "MT-CO1", "IGF1", "ARL4C", "NPC2", "CTSC", "MT-CO2", "CTSZ",
    "MAFB", "CD209", "CD163L1", "PSAP", "CTSD", "LIPA", "JUNB", "MRC1", "TPT1", "MT-ATP6", "MT-CYB", "FOLR2",
    "LILRB5", "AXL", "FPR3", "CEBPD", "JUN", "RNASE6", "CTSB", "KCTD12", "RNASE1", "DUSP1", "F13A1", "LPAR6",
    "DNASE1L3", "RGS10", "RPL13", "FUCA1", "PDK4", "TGFBI", "PLD3", "GRN", "HLA-DPA1", "HLA-DPB1", "ZFP36L2",
    "CXCL12", "RNASET2", "TMSB4X", "FRMD4A", "NR4A2", "MCL1", "ZFP36L1", "CSF1R", "AHR", "CD4", "ADAP2", "GATM",
    "PTMA", "GPR34", "LMNA", "GNAS", "RPLP2", "NRP1", "FRMD4B", "LACC1", "ATF3", "HLA-DMB", "WWP1", "MS4A4A",
    "LINC00996", "PRNP", "SDC3", "CD163", "PABPC1", "TPP1"
]

# Score the gene set
sc.tl.score_genes(adata1, gene_list=l1_genes, score_name='intestineLYVE1score')

In [ ]:
adata1

In [ ]:
sc.pl.umap(
    adata1,
    color= ['intestineLYVE1score', 
        'leiden_res_0_8'   
        ],
    cmap='viridis',           
    size=150,                 
    title='Macrophage Signature Score',
    vmin = 0,
    vmax = 0.5,
    save = 'intestine_score.png'
)